In [25]:
import duckdb
import pandas as pd

# Load the CSV into a DataFrame
df = pd.read_csv("data/frag_raw.csv")

# Connect to DuckDB (in-memory)
con = duckdb.connect()

# Register the DataFrame as a DuckDB table
con.register('fragrances', df)

# Run your SQL query (adapted for DuckDB syntax)
query = '''
SELECT
    TRIM(SUBSTRING(description, 1, instr(description, 'by')-1)) AS name,
    TRIM(SUBSTRING(description, instr(description, 'by')+3, instr(description, 'is a')-instr(description, 'by')-3)) AS brand,
    CASE
        WHEN description LIKE '%for women and men%' THEN 'unisex'
        WHEN description LIKE '%for women%' THEN 'women'
        WHEN description LIKE '%for men%' THEN 'men'
        ELSE NULL
    END AS gender,
    CAST(REPLACE(rating_count, ',', '') AS INTEGER) AS rating_count,
    REPLACE(REPLACE(REPLACE(main_accords, '[', ''), ']', ''), "'", '') AS main_accords,
    REPLACE(regexp_extract(description, 'Top notes are (.*?);', 1), ' and', ',') AS top_notes,
    REPLACE(regexp_extract(description, 'middle notes are (.*?);', 1), ' and', ',') AS mid_notes,
    REPLACE(regexp_extract(description, 'base notes are (.*?)\\.', 1), ' and', ',') AS base_notes,
    description,
    url
FROM fragrances
'''

result = con.execute(query).df()
print(result.head())

BinderException: Binder Error: Referenced column "'" not found in FROM clause!
Candidate bindings: "url"

LINE 12: ...    REPLACE(REPLACE(REPLACE(main_accords, '[', ''), ']', ''), "'", '') AS main_accords,
                                                                          ^

In [16]:
from pyspark.sql import SparkSession

# Initialize a Spark session
spark = SparkSession.builder.appName("FragranceDataCleaning").getOrCreate()

# Define the file path
file_path = "D:/Users/Wednesday/Documents/GitHub/sniffers/data/frag_raw.csv"

try:
    # Load the raw CSV into a Spark DataFrame using the specified path
    frag_df = spark.read.csv(file_path, header=True, inferSchema=True)

    # Register the DataFrame as a temporary view for Spark SQL
    frag_df.createOrReplaceTempView("fragrances")

    frag_df.show(10, truncate=False)



    # Clean the data using a single Spark SQL query
    cleaned_df = spark.sql("""
        SELECT
            SUBSTRING_INDEX(SUBSTRING_INDEX(description, 'by', 1), 'is a', 1) AS name,
            SUBSTRING_INDEX(SUBSTRING_INDEX(description, 'by', -1), 'is a', 1) AS brand,
            CASE
                WHEN name LIKE '%for women and men' THEN 'unisex'
                WHEN name LIKE '%for women' THEN 'women'
                WHEN name LIKE '%for men' THEN 'men'
                ELSE NULL
            END AS gender,
            CAST(
                REGEXP_REPLACE(rating_count, ',', '') AS INT
            ) AS rating_count,
            REPLACE(REPLACE(REPLACE(main_accords, '[', ''), ']', ''),"'", "") AS main_accords,
            REGEXP_REPLACE(REGEXP_EXTRACT(description, 'Top notes are (.*?);', 1), " and", ",") AS top_notes,
            REGEXP_REPLACE(REGEXP_EXTRACT(description, 'middle notes are (.*?);', 1), " and", ",") AS mid_notes,
            REGEXP_REPLACE(REGEXP_EXTRACT(description, 'base notes are (.*?)\\.', 1), " and", ",") AS base_notes,
            description,
            url
        FROM fragrances
    """)

    # Show the cleaned DataFrame
    print("Cleaned DataFrame:")
    cleaned_df.show(truncate=False)

    # Print the schema to show the new data types
    print("Cleaned DataFrame Schema:")
    cleaned_df.printSchema()

    # # Save the cleaned data to a new CSV file
    # cleaned_df.write.csv("frag_clean.csv", header=True, mode="overwrite")

except Exception as e:
    print(f"An error occurred: {e}")
    print("Please ensure the file path is correct and accessible.")

finally:
    # Stop the Spark session
    spark.stop()

+----------------------------------------+-----------------+------+------------+---------------------------------------------------------------------------------------------------------------+---------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------+
|name                                    |gender           |rating|rating_count|main_accords                                                                                                   |perfumers|description                                                                                                                                                                                                                                          